# Obtaining FASTA files from Entrez Database
The BioPython library provides a function to query the NCBI database for specific species and genes of interest.

In [2]:
from Bio import Entrez, SeqIO # Get this library from `pip install biopython`
from io import StringIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import time
import pandas as pd
import os
from typing import List

## Diverse Bacteria

Get the bacteria needed from bacteria_metadata.csv

In [6]:
metadata = pd.read_csv("./Data/bacteria_metadata.csv")
metadata.head()

,species,gram_stain,pathogenicity,notes
0,Acinetobacter baumannii,Gram-negative,Pathogenic,NaN
1,Acinetobacter baylyi,Gram-negative,Non-pathogenic,Environmental
2,Acinetobacter radioresistens,Gram-negative,Non-pathogenic,Environmental
3,Actinomyces israelii,Gram-positive,Pathogenic,NaN
4,Alteromonas macleodii,Gram-negative,Non-pathogenic,Marine


In [8]:
bacteria = metadata['species'].tolist()
print(len(bacteria), "bacterial species total")

100 bacterial species total


In [29]:
# Genes of interest
genes = ["rpoD", "gyrA"]

In [7]:
def QueryEntrezDatabase(species_list: List[str], gene_list: List[str], output_folder="./"):
    """ Main function to query from NCBI database to obtain genes across bacterial species. Note this only searches for the "gene" terms, not rRNA.

    Args:
        species_list (List[str]): List of strings representing bacterial species names
        gene_list (List[str]): List of genes you want to get from the database
        output_folder (str, optional): Defaults to "./". Specify the folder/directory you want the sequences to be saved to
    """
    
    Entrez.email = "" # your-email-here
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    
    # Save each gene to its own fasta file, iterate over genes first then species next
    for gene in gene_list:
        
        # Name the file based on the gene name
        output_filename = output_folder + f"{gene}_cds_all_bacteria.fasta"
        
        for species in species_list:
            
            # We want only the DNA sequence of the full CDS of this gene across species
            database = "nucleotide"
            search_term = f"{species}[Organism] AND {gene}[Gene Name] AND CDS[Feature] NOT UNVERIFIED[Text]"
            retmax_search = "50" # Increase retmax to potentially find multiple CDS for a gene in a species
            rettype_fetch = "gb" # Fetch in GenBank format to get feature information
            retmode_fetch = "text"
            
            found_cds_for_species = False

            try:
                # Perform the search
                handle = Entrez.esearch(db=database, term=search_term, retmax=retmax_search)
                search_results = Entrez.read(handle)
                handle.close()
                id_list = search_results["IdList"]

                # If we have search results:
                if id_list:
                    print(f"Found {len(id_list)} records with IDs: {id_list}")

                    # Loop thru each record
                    for record_id in id_list:
                        
                        if not found_cds_for_species:
                            # Use Entrez query to the database
                            try:
                                handle = Entrez.efetch(db=database, id=record_id, rettype=rettype_fetch, retmode=retmode_fetch)
                                genbank_record = handle.read()
                                handle.close()

                                for seq_record in SeqIO.parse(StringIO(genbank_record), "genbank"):
                                    organism = seq_record.annotations.get("organism")
                                    
                                    # Ensure the organism in the record matches the queried species
                                    if organism == species: 
                                        
                                        for feature in seq_record.features:
                                            
                                            # Only get full CDS, not partial
                                            if feature.type == "CDS":
                                                if "partial" not in feature.qualifiers:
                                                    
                                                    gene_name_qualifier = feature.qualifiers.get('gene')
                                                    
                                                    if gene_name_qualifier and gene_name_qualifier[0] == gene:
                                                        try:
                                                            cds_seq = feature.extract(seq_record.seq)
                                                            fasta_description = organism
                                                            fasta_id = f"{organism.replace(' ', '_')}_{gene}"
                                                            fasta_record = SeqRecord(cds_seq, id=fasta_id, description=fasta_description)

                                                            with open(output_filename, "a") as outfile:
                                                                SeqIO.write(fasta_record, outfile, "fasta")
                                                            print(f"      - Extracted full CDS for '{gene}' in '{organism}' (Record ID: {seq_record.id}).")
                                                            found_cds_for_species = True
                                                            
                                                            # Break inner feature loop after finding one full CDS for the target gene
                                                            break 
                                                        except Exception as e:
                                                            pass # Don't log anything
                                                            #print(f"Error extracting CDS from record {seq_record.id}: {e}")
                                                else:
                                                    print(f"      - Skipping partial CDS from record {seq_record.id}")

                            except Exception as e:
                                print(f"Error fetching record with ID {record_id}: {e}")

                        # If we found the CDS, break the query
                        else:
                            #print(f"Full CDS already found for '{gene}' in '{species}'. Skipping remaining records.")
                            break
                else:
                    print(f"    No records found for '{gene}' in '{species}' with full CDS and excluding 'UNVERIFIED'.")

            except Exception as e:
                print(f"  An error occurred during search for '{gene}' in '{species}': {e}")

            time.sleep(1) # Wait 1 sec after each species query

        time.sleep(5) # Wait 5 secs before starting the next gene queries

In [43]:
base_folder = "./Data/"
QueryEntrezDatabase(bacteria, genes, base_folder)

Found 50 records with IDs: ['2936345732', '2936342332', '2701990796', '2701990689', '2701990548', '2701985715', '2701985048', '2701984331', '2701983559', '2701982801', '2701964417', '2701963901', '2701963422', '2701961106', '2701960961', '2701960816', '2701921414', '2701920960', '2701920345', '2701919821', '2935869790', '2935869777', '2935869757', '2935869680', '2935869646', '2935869631', '2935869617', '2935869594', '2935869573', '2935869550', '2935869520', '2935869501', '2935869480', '2935869454', '2935869383', '2935868604', '2935867439', '2935867386', '2935867292', '2935867257', '2935867192', '2935867130', '2935867065', '2935867012', '2935866419', '2935865451', '2935865379', '2935865197', '2935864200', '2935863999']
      - Extracted full CDS for 'rpoD' in 'Acinetobacter baumannii' (Record ID: CP178253.1).
Found 23 records with IDs: ['491068042', '1487826061', '651316845', '2578388561', '2577663963', '2569875478', '1815895063', '1815863699', '1815862588', '971985882', '1815937004', '

## Closely-Related Bacteria

In [4]:
metadata_related = pd.read_csv("./Data/related/related_bacterial_species.csv")
metadata_related.head()

,Genus,Species
0,Escherichia,Escherichia coli
1,Escherichia,Escherichia albertii
2,Escherichia,Escherichia fergusonii
3,Escherichia,Escherichia hermannii
4,Escherichia,Escherichia marmotae


In [5]:
related_bacteria = metadata_related["Species"].tolist()
print(len(related_bacteria), "bacterial species total")

100 bacterial species total


In [8]:
out_folder = "./Data/related/"
genes = ["rpoD", "gyrA"]


QueryEntrezDatabase(related_bacteria, genes, out_folder)

Found 50 records with IDs: ['2957588341', '2700745328', '2700743737', '2700743467', '2700742788', '2700741744', '2700740880', '2700740479', '2700740058', '2700739842', '2700739467', '2700739199', '2700739014', '2700738593', '2700738148', '2700737712', '2700737219', '2700736893', '2700736603', '2700736173', '2700735810', '2700735680', '2700735206', '2700734739', '2700734519', '2700734231', '2700733914', '2700733620', '2700733255', '2700731724', '2700731296', '2700730989', '2700730548', '2700730309', '2700730056', '2700729846', '2700729495', '2700729288', '2700728744', '2700728464', '2700728156', '2700727821', '2700727158', '2700726759', '2700726520', '2700726246', '2700725763', '2700725742', '2700725253', '2700724088']
      - Extracted full CDS for 'rpoD' in 'Escherichia coli' (Record ID: CP187284.1).
Found 50 records with IDs: ['1356470383', '1356472090', '1759504095', '1585998072', '1535555144', '1528231152', '1356470835', '1355929686', '935479443', '1358833571', '1535599210', '15355